# Linux Service Won't Start — A Systematic `systemd` Troubleshooting Lab

**Audience:** AI/ML engineers, backend engineers, DevOps/MLOps engineers

## Objective

Learn to diagnose a Linux service that will not start by collecting evidence, identifying the failing layer, fixing the root cause, and verifying the result.

The goal is **not** to memorize `systemctl` commands. The goal is to understand what each command proves.

### Startup mental model

```text
systemd
  ↓
unit file
  ↓
dependencies
  ↓
user / permissions / environment / limits
  ↓
ExecStart
  ↓
application initialization
  ↓
healthy long-running process
```

A failure can happen at any layer.

# 1. Essential vocabulary

### systemd
A Linux init system and service manager used by many modern distributions.

### Unit
An object managed by systemd. Common types include `.service`, `.socket`, `.timer`, `.mount`, and `.target`.

### Service unit
A `.service` unit describes how a process should be started, stopped, restarted, and supervised.

### Unit file
The configuration file for a unit. Common locations are:

- `/etc/systemd/system/` — administrator-created/overridden units
- `/usr/lib/systemd/system/` or `/lib/systemd/system/` — package-provided units
- `/run/systemd/system/` — runtime-generated units

### PID 1
The first userspace process. On a normal systemd host, systemd is PID 1.

### D-Bus
The communication mechanism normally used by `systemctl` to communicate with the systemd manager.

**Critical distinction:**

```text
systemctl installed
    ≠
systemd running as PID 1
    ≠
systemctl able to manage services here
```

This distinction matters heavily in Docker/container environments.

# 2. Diagnose the environment first

Before debugging the service, establish what kind of machine you are actually on.

This is especially important for AI engineers because the same application may run on:

- a laptop
- a VM
- a cloud instance
- a Docker container
- Kubernetes
- a managed inference platform

If PID 1 is not systemd, `systemctl` may not be the correct process-management interface.

In [ ]:
import os
import platform
import shutil
import subprocess

print("OS:", platform.platform())
print("Kernel:", platform.release())

pid1 = subprocess.run(
    ["ps", "-p", "1", "-o", "pid,comm,args="],
    text=True,
    capture_output=True
)
print("\nPID 1:")
print(pid1.stdout.strip())

print("\nImportant commands:")
for command in ["systemctl", "journalctl", "systemd-analyze", "ss", "lsof", "netstat"]:
    print(f"{command:16} -> {shutil.which(command) or 'NOT INSTALLED'}")

## How to interpret the environment check

Ask two independent questions:

1. **Is the tool installed?**
2. **Is the relevant daemon/supervisor actually running?**

A minimal container can contain the `systemctl` executable while PID 1 is something else such as `supervisord`.

In that case, installing more systemd packages will not magically make the container a systemd host.

# 3. Step 1 — `systemctl status`

### Command

```bash
systemctl status <service>
```

Example:

```bash
systemctl status nginx
```

### What it tells you

It gives a compact view of:

- whether the unit is loaded
- current state
- main PID
- recent log lines
- exit status
- startup failure hints

### Common states

| State | Meaning |
|---|---|
| `active (running)` | Process is running |
| `inactive (dead)` | Not running; not necessarily an error |
| `failed` | A failure was recorded |
| `activating` | Startup is still in progress |

**Important:** `inactive` and `failed` are not synonymous.

In [ ]:
# Read-only test.
# The service may not exist, and the current environment may not run systemd.
!systemctl status sshd

### Interpreting the tested result

If you receive:

```text
System has not been booted with systemd as init system (PID 1)
Failed to connect to bus
```

the immediate problem is **the execution environment**, not necessarily `sshd`.

On a real systemd host, the same command would normally show the unit state and recent service information.

# 4. Step 2 — `journalctl`

### Command

```bash
journalctl -u <service> -xe
```

Breakdown:

- `journalctl` — reads the systemd journal
- `-u` — filter by unit
- `-x` — add explanatory information when available
- `-e` — jump toward the end

Useful variations:

```bash
journalctl -u <service> --since "1 hour ago"
journalctl -u <service> -n 100 --no-pager
journalctl -u <service> -f
```

`-f` follows new entries, similar to `tail -f`.

In [ ]:
# Read-only test.
!journalctl -u sshd -xe

### Important interpretation

`No journal files were found` does not automatically mean "the service has no logs."

It can mean the environment does not have a running/persistent journald setup.

On a real systemd host, service-specific journal entries are often the strongest evidence for the actual failure.

# 5. Step 3 — Validate the unit file

### Command

```bash
systemd-analyze verify /path/to/my-service.service
```

This checks the **unit definition**, rather than asking the running service for its status.

Typical problems include:

- invalid directives
- missing referenced units
- invalid executable configuration
- dependency/configuration problems

A useful troubleshooting habit is to test both a known-good unit and a deliberately broken unit.

In [ ]:
from pathlib import Path
import subprocess
import tempfile

def run_command(command):
    p = subprocess.run(command, shell=True, text=True, capture_output=True)
    print("$", command)
    print("exit code:", p.returncode)
    print((p.stdout + p.stderr).strip() or "[no output]")
    print()

with tempfile.TemporaryDirectory() as td:
    good = Path(td) / "notebook-good.service"
    good.write_text("""[Unit]
Description=Notebook Good Demo

[Service]
Type=oneshot
ExecStart=/bin/true
""")
    run_command(f"systemd-analyze verify {good}")

    bad = Path(td) / "notebook-bad.service"
    bad.write_text("""[Unit]
Description=Notebook Bad Demo

[Service]
Type=oneshot
ExecStart=/does/not/exist
""")
    run_command(f"systemd-analyze verify {bad}")

### Why this test is better than only running `systemd-analyze verify sshd`

`systemd-analyze verify sshd` treats `sshd` as an argument that should resolve to a valid unit-file input. That is not a useful success-path demonstration in an environment where the unit file is unavailable.

The temporary lab above gives us:

- a known-good unit
- a known-bad unit
- a real exit code
- real validation behavior

This is how you should design reproducible troubleshooting experiments.

# 6. Step 4 — Inspect the effective unit

```bash
systemctl cat <service>
```

This helps answer:

> "What configuration is systemd actually using?"

Look for:

```ini
[Unit]
After=
Requires=

[Service]
User=
Group=
WorkingDirectory=
Environment=
EnvironmentFile=
ExecStart=
Restart=

[Install]
WantedBy=
```

Drop-in overrides are especially important because an administrator or package can override the configuration you expected.

In [ ]:
!systemctl cat sshd

# 7. Step 5 — Inspect dependencies

```bash
systemctl list-dependencies <service>
```

This is useful when your service relies on other units.

Two commonly confused directives are:

```ini
Requires=database.service
After=database.service
```

`Requires=` expresses a dependency relationship.

`After=` expresses startup ordering.

**`After=` alone does not mean "start the other service."**

In [ ]:
!systemctl list-dependencies sshd

# 8. Step 6 — Verify the executable and service user

If logs show:

```text
status=203/EXEC
```

investigate `ExecStart`.

For example:

```ini
ExecStart=/opt/myapp/.venv/bin/uvicorn app:app --port 8000
```

Check:

```bash
ls -l /opt/myapp/.venv/bin/uvicorn
file /opt/myapp/.venv/bin/uvicorn
command -v uvicorn
```

If the service runs as `mlservice`, reproduce the command as that identity when appropriate:

```bash
sudo -u mlservice /opt/myapp/.venv/bin/uvicorn ...
```

This can expose permission, environment, and import errors hidden by the service wrapper.

In [ ]:
import shutil
import os

for command in ["python", "python3", "bash", "uvicorn"]:
    print(f"{command:10} -> {shutil.which(command) or 'NOT FOUND'}")

print("\nCurrent directory:", os.getcwd())
print("Readable:", os.access(os.getcwd(), os.R_OK))

# 9. Step 7 — Check for port conflicts

A perfectly valid service can fail because another process already owns its port.

Typical application error:

```text
Address already in use
```

Use:

```bash
ss -tulpn
```

For TCP port 8000:

```bash
ss -ltnp | grep ':8000'
```

Flags:

- `-t` — TCP
- `-u` — UDP
- `-l` — listening sockets
- `-p` — process information when permitted
- `-n` — numeric addresses/ports

For AI systems, this commonly affects FastAPI, vLLM, Triton, model workers, and monitoring services.

In [ ]:
# Read-only network inspection.
!ss -tulpn

### Tested result note

The original notebook recorded `ss: not found`, but a fresh test in the current environment shows that `ss` is installed and returns real listening sockets.

Therefore this notebook documents the command as **environment-dependent**, rather than claiming that `ss` is universally missing.

If it is absent:

```bash
command -v ss
```

can confirm that fact before you decide on a package/fallback.

# 10. Step 8 — Check resources

These commands do not require systemd.

### Disk

```bash
df -h
```

Look for filesystems near 100%.

### Memory

```bash
free -h
```

Pay attention to `available`, not just `free`.

### Process/file limits

```bash
ulimit -a
```

Useful limits include:

- open files (`nofiles`)
- max user processes
- stack size
- locked memory
- core dump size

In [ ]:
print("### DISK")
!df -h

print("\n### MEMORY")
!free -h

print("\n### RESOURCE LIMITS")
!ulimit -a

# 11. Step 9 — Security controls

If ordinary Unix permissions look correct but access is denied, investigate mandatory access controls.

### SELinux

```bash
getenforce
sudo ausearch -m avc -ts recent
```

### AppArmor

```bash
sudo aa-status
```

Do not disable security controls as your first troubleshooting action.

The objective is to identify the denied operation and correct the policy, context, or service configuration.

# 12. Step 10 — Restart loops and `start-limit-hit`

A service can crash immediately and be restarted repeatedly.

For example:

```ini
Restart=on-failure
```

can turn one application crash into a restart loop.

You may eventually see a start-limit-related failure.

```bash
systemctl reset-failed <service>
```

clears recorded failure state.

**It does not repair the application.**

Always investigate the earlier log/error first.

# 13. Common failure → hypothesis → evidence

| Symptom | Likely area | First evidence |
|---|---|---|
| `203/EXEC` | executable/path/permission | `systemctl cat`, `ls -l`, executable test |
| `127` | command not found | `command -v`, PATH |
| `Permission denied` | permissions/security | ownership, service user, SELinux/AppArmor |
| `Address already in use` | network | `ss -ltnp` |
| `No such file or directory` | path/config | verify referenced files |
| `status=1/FAILURE` | application | manual execution + stderr |
| `start-limit-hit` | crash loop | earlier journal entries |
| `Unit not found` | wrong name/missing unit | `systemctl list-unit-files` |
| `Failed to connect to bus` | systemd unavailable | inspect PID 1/environment |
| starts then exits | application/config/type | logs + process behavior |

These are **hypotheses**, not diagnoses. Evidence must confirm them.

# 14. Safe change workflow

When you need to change a real service:

### 1 — Inspect

```bash
systemctl status my-service
systemctl cat my-service
journalctl -u my-service --since "30 minutes ago"
```

### 2 — Edit only what is necessary

### 3 — Validate

```bash
systemd-analyze verify /etc/systemd/system/my-service.service
```

### 4 — Reload unit definitions if the unit file changed

```bash
sudo systemctl daemon-reload
```

### 5 — Restart

```bash
sudo systemctl restart my-service
```

### 6 — Verify

```bash
systemctl status my-service
journalctl -u my-service -n 100 --no-pager
```

### 7 — Verify application health

A process being `active` is not enough for an AI API. Also test:

- health endpoint
- model availability
- database connectivity
- queue connectivity
- inference request
- GPU/resource behavior

# 15. AI/ML Engineer incident example

Suppose you have:

```ini
[Unit]
Description=FastAPI Model Server
After=network.target

[Service]
User=mlservice
WorkingDirectory=/opt/model-api
EnvironmentFile=/etc/my-model.env
ExecStart=/opt/model-api/.venv/bin/uvicorn app:app --host 0.0.0.0 --port 8000
Restart=on-failure

[Install]
WantedBy=multi-user.target
```

It fails to start.

Do not immediately reinstall Python.

Ask:

1. Is systemd running?
2. Is the unit loaded?
3. What does `systemctl status` show?
4. What does `journalctl` show?
5. Does the unit validate?
6. Does `ExecStart` exist?
7. Can `mlservice` execute it?
8. Is the environment file readable?
9. Does `app:app` import?
10. Is port 8000 occupied?
11. Is disk/memory sufficient?
12. Is SELinux/AppArmor involved?

This is the troubleshooting mindset expected from an engineer who owns production AI systems.

# 16. Container lesson

A common mistake is assuming:

> "I have `systemctl`, so this must be a systemd machine."

Not necessarily.

A container can contain systemd binaries while PID 1 is:

```text
supervisord
```

or another process supervisor.

If systemd is not PID 1, use the process manager/orchestrator that actually owns the workload.

For Docker/Kubernetes, the correct troubleshooting layer may be:

```text
docker logs
docker inspect
docker exec
kubectl logs
kubectl describe pod
kubectl get events
```

rather than `systemctl`.

# 17. Command cheat sheet with purpose

| Command | Why you run it |
|---|---|
| `systemctl status X` | current unit state + recent clues |
| `journalctl -u X` | service-specific logs |
| `systemctl cat X` | effective unit configuration |
| `systemctl list-dependencies X` | dependency graph |
| `systemd-analyze verify FILE` | validate unit definition |
| `systemctl daemon-reload` | make systemd reread changed unit definitions |
| `systemctl restart X` | retest after a change |
| `systemctl reset-failed X` | clear recorded failed state |
| `ss -ltnp` | find listening TCP ports/processes |
| `df -h` | disk capacity |
| `free -h` | memory availability |
| `ulimit -a` | process/resource limits |
| `ls -l PATH` | ownership/permission/executable bit |
| `sudo -u USER CMD` | reproduce execution as service user |

**Memorize the purpose, not just the syntax.**

# 18. Incident report template

```text
Date/time:
Host:
Environment:
Service:
Version/deployment:

SYMPTOM
What exactly failed?

INITIAL EVIDENCE
systemctl status:
journalctl:
systemctl cat:
systemd-analyze verify:

ENVIRONMENT
PID 1:
OS:
Container / VM / bare metal:

APPLICATION
ExecStart:
Service user:
WorkingDirectory:
Environment/config:
Manual execution result:

NETWORK
Expected port:
Conflict:
Listening process:

RESOURCES
Disk:
Memory:
Open files/process limits:

SECURITY
SELinux:
AppArmor:
Permissions:

ROOT CAUSE
What actually caused the failure?

FIX
What changed?

VERIFICATION
How was the fix verified?
How long was stability observed?

PREVENTION
What monitoring/test/deployment/documentation change should be added?
```

# 19. Final decision tree

```text
Service won't start
       |
       v
Is systemd actually running?
       |
   +---+---+
   |       |
  NO      YES
   |       |
Check      systemctl status
container      |
supervisor     v
            failed?
               |
               v
           journalctl
               |
               v
       classify the error
       /      |       |      \
    EXEC   CONFIG    PORT   PERMISSION
      |       |        |       |
   binary   app      ss      user/
    path   config    port    security
       \      |       |      /
             FIX
              |
              v
      daemon-reload*
              |
              v
          restart
              |
              v
       process + health
```

`daemon-reload` is needed when unit definitions changed; it is not a universal repair command.

# 20. Reproducibility test record

The commands in the original notebook were re-tested during this revision.

Observed in the current execution environment:

- `systemctl --version` is installed.
- `systemctl is-system-running` reports `offline`.
- PID 1 is `supervisord`, not systemd.
- `systemctl status`, `systemctl cat`, and `systemctl list-dependencies` cannot connect to the systemd manager here.
- `journalctl -u sshd -xe` reports no journal files.
- A temporary valid unit passes `systemd-analyze verify`.
- A temporary invalid unit with a nonexistent `ExecStart` is rejected.
- `ss -tulpn` is installed and returns real listening sockets.
- `df -h`, `free -h`, and `ulimit -a` execute successfully.

The notebook deliberately separates **tested behavior in this environment** from **expected behavior on a real systemd host**.

In [ ]:
# Lightweight re-test cell.
# This is intentionally read-only.
import subprocess

commands = [
    "systemctl --version | head -n 1",
    "systemctl is-system-running",
    "ps -p 1 -o pid,comm,args=",
    "command -v ss",
    "df -h",
    "free -h",
]

for command in commands:
    p = subprocess.run(command, shell=True, text=True, capture_output=True)
    print(f"\n$ {command}")
    print(f"exit={p.returncode}")
    print((p.stdout + p.stderr).strip() or "[no output]")